# 🚗 Notebook 1: Vehicle Detection using YOLOv8
**Smart Cities: AI-Based Vehicle Tracking and License Plate Recognition System**

---
### 📌 Is notebook mein hum:
- YOLOv8 model load karenge (pretrained on COCO)
- Video se frames extract karenge (OpenCV)
- Vehicles detect karenge: Car, Bike, Truck, Bus
- Detected vehicles pe bounding boxes draw karenge
- Output video save karenge

---

## Step 1: Install Required Libraries

In [ ]:
# Sab required packages install karo
!pip install ultralytics opencv-python-headless matplotlib Pillow
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

## Step 2: Import Libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
import os
import time

print('✅ Libraries imported successfully!')

## Step 3: Load YOLOv8 Pretrained Model

> YOLOv8 COCO dataset pe train hua hai jisme 80 classes hain.
> Hum sirf vehicle classes use karenge: car, motorcycle, bus, truck

In [ ]:
# YOLOv8n (nano) model load karo - sabse fast, production ke liye yolov8m ya yolov8l use karo
model = YOLO('yolov8n.pt')  # Automatically download hoga pehli baar

print('✅ YOLOv8 model loaded successfully!')
print(f'Model type: {type(model)}')

# COCO classes mein se vehicle classes
VEHICLE_CLASSES = {
    2: 'car',
    3: 'motorcycle',
    5: 'bus',
    7: 'truck'
}

# Har vehicle class ke liye alag color
CLASS_COLORS = {
    'car':        (0, 255, 0),    # Green
    'motorcycle': (255, 165, 0),  # Orange
    'bus':        (0, 0, 255),    # Blue
    'truck':      (0, 255, 255),  # Cyan
}

print(f'\n🚗 Tracking vehicle classes: {list(VEHICLE_CLASSES.values())}')

## Step 4: Helper Functions — Detection & Visualization

In [ ]:
def detect_vehicles_in_frame(frame, model, conf_threshold=0.4):
    """
    Ek frame mein vehicles detect karo.
    
    Args:
        frame: OpenCV BGR image
        model: YOLOv8 model
        conf_threshold: Minimum confidence score
    
    Returns:
        detections: List of dicts {bbox, class_id, class_name, confidence}
        annotated_frame: Frame with bounding boxes drawn
    """
    results = model(frame, verbose=False)[0]
    detections = []
    annotated_frame = frame.copy()
    
    for box in results.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        
        # Sirf vehicle classes aur high confidence
        if class_id not in VEHICLE_CLASSES:
            continue
        if confidence < conf_threshold:
            continue
        
        class_name = VEHICLE_CLASSES[class_id]
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # Detection store karo
        detection = {
            'bbox': (x1, y1, x2, y2),
            'class_id': class_id,
            'class_name': class_name,
            'confidence': confidence,
            'centroid': ((x1 + x2) // 2, (y1 + y2) // 2)
        }
        detections.append(detection)
        
        # Bounding box draw karo
        color = CLASS_COLORS[class_name]
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
        
        # Label with confidence
        label = f'{class_name} {confidence:.2f}'
        label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated_frame,
                      (x1, y1 - label_size[1] - 10),
                      (x1 + label_size[0] + 5, y1),
                      color, -1)
        cv2.putText(annotated_frame, label,
                    (x1 + 3, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (0, 0, 0), 2)
        
        # Centroid dot
        cx, cy = detection['centroid']
        cv2.circle(annotated_frame, (cx, cy), 5, color, -1)
    
    return detections, annotated_frame


def draw_detection_stats(frame, detections, frame_num, fps=0):
    """Frame pe stats overlay karo."""
    # Count per class
    counts = {}
    for d in detections:
        counts[d['class_name']] = counts.get(d['class_name'], 0) + 1
    
    # Stats box (top-left corner)
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (220, 140), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    
    cv2.putText(frame, f'Frame: {frame_num}', (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    cv2.putText(frame, f'FPS: {fps:.1f}', (15, 52),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    cv2.putText(frame, f'Total: {len(detections)}', (15, 74),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)
    
    y_offset = 96
    for cls, count in counts.items():
        color = CLASS_COLORS.get(cls, (255, 255, 255))
        cv2.putText(frame, f'  {cls}: {count}', (15, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        y_offset += 18
    
    return frame


def show_frame_matplotlib(frame, title='Detection Result'):
    """Jupyter mein frame display karo."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(14, 8))
    plt.imshow(rgb)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


print('✅ Helper functions defined!')

## Step 5: Test on a Single Image (Sanity Check)

In [ ]:
# ---- Option A: Online image se test karo ----
import urllib.request

test_image_url = 'https://ultralytics.com/images/bus.jpg'
test_image_path = 'test_image.jpg'

urllib.request.urlretrieve(test_image_url, test_image_path)
print(f'✅ Test image downloaded: {test_image_path}')

# Frame load karo
test_frame = cv2.imread(test_image_path)

# Detection run karo
start = time.time()
detections, annotated = detect_vehicles_in_frame(test_frame, model, conf_threshold=0.3)
elapsed = time.time() - start

# Stats overlay
annotated = draw_detection_stats(annotated, detections, frame_num=1, fps=1/elapsed)

print(f'\n📊 Detection Results:')
print(f'   Total vehicles detected: {len(detections)}')
print(f'   Inference time: {elapsed*1000:.1f} ms')
for d in detections:
    print(f'   → {d["class_name"]:12s} | conf: {d["confidence"]:.3f} | bbox: {d["bbox"]} | centroid: {d["centroid"]}')

# Display
show_frame_matplotlib(annotated, '🚗 Vehicle Detection — Single Image Test')

## Step 6: Run Detection on Video File

> **Sample videos**: https://drive.google.com/drive/folders/11n5u1B3BAppISJl7OgLNII3shY9FnLsV
>
> Video download karke `VIDEO_PATH` update karo.

In [ ]:
# ============================================================
#  CONFIG — Apna video path yahan set karo
# ============================================================
VIDEO_PATH   = 'traffic_video.mp4'   # <-- apna video path
OUTPUT_PATH  = 'output_detection.mp4'
CONF_THRESH  = 0.4
MAX_FRAMES   = 300   # Testing ke liye limit; full video = None
DISPLAY_EVERY = 30   # Jupyter mein har N frame display karo

# ============================================================
# Agar video nahi hai: webcam use karo (0 = default cam)
# VIDEO_PATH = 0
# ============================================================

print(f'🎬 Video source: {VIDEO_PATH}')

In [ ]:
def process_video_detection(video_path, output_path, model,
                             conf_threshold=0.4,
                             max_frames=None,
                             display_every=30):
    """
    Video process karo: har frame mein vehicles detect karo.
    
    Returns:
        all_detections: List of per-frame detection dicts
        stats: Summary statistics
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f'❌ Error: Video open nahi hua — {video_path}')
        print('   Sample video download karo: https://drive.google.com/drive/folders/11n5u1B3BAppISJl7OgLNII3shY9FnLsV')
        return [], {}
    
    # Video properties
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f'📹 Video Info:')
    print(f'   Resolution : {width}x{height}')
    print(f'   FPS        : {fps:.1f}')
    print(f'   Total Frames: {total_frames}')
    print(f'   Duration   : {total_frames/fps:.1f}s')
    
    # Output video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    all_detections = []
    frame_num = 0
    process_limit = max_frames if max_frames else total_frames
    
    print(f'\n⚙️  Processing {process_limit} frames...')
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and frame_num >= max_frames:
            break
        
        t0 = time.time()
        detections, annotated = detect_vehicles_in_frame(frame, model, conf_threshold)
        frame_fps = 1.0 / (time.time() - t0 + 1e-9)
        
        # Stats draw
        annotated = draw_detection_stats(annotated, detections, frame_num, frame_fps)
        
        # Save
        out.write(annotated)
        
        # Record
        all_detections.append({
            'frame': frame_num,
            'detections': detections,
            'count': len(detections)
        })
        
        # Jupyter preview
        if frame_num % display_every == 0:
            print(f'  Frame {frame_num:4d}/{process_limit} | Vehicles: {len(detections)} | FPS: {frame_fps:.1f}')
            if frame_num % (display_every * 5) == 0:
                show_frame_matplotlib(annotated, f'Frame {frame_num} — Detection Preview')
        
        frame_num += 1
    
    cap.release()
    out.release()
    
    # Summary stats
    total_detections = sum(d['count'] for d in all_detections)
    stats = {
        'total_frames_processed': frame_num,
        'total_detections': total_detections,
        'avg_vehicles_per_frame': total_detections / max(frame_num, 1)
    }
    
    print(f'\n✅ Processing complete!')
    print(f'   Frames processed : {frame_num}')
    print(f'   Total detections : {total_detections}')
    print(f'   Avg per frame    : {stats["avg_vehicles_per_frame"]:.2f}')
    print(f'   Output saved to  : {output_path}')
    
    return all_detections, stats


# === RUN ===
all_detections, stats = process_video_detection(
    VIDEO_PATH, OUTPUT_PATH, model,
    conf_threshold=CONF_THRESH,
    max_frames=MAX_FRAMES,
    display_every=DISPLAY_EVERY
)

## Step 7: Detection Analytics — Graphs

In [ ]:
if all_detections:
    # Per-frame vehicle count
    frame_nums  = [d['frame'] for d in all_detections]
    frame_counts = [d['count'] for d in all_detections]
    
    # Per-class total counts
    class_totals = {}
    for fd in all_detections:
        for det in fd['detections']:
            cls = det['class_name']
            class_totals[cls] = class_totals.get(cls, 0) + 1
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('🚗 Vehicle Detection Analytics', fontsize=16, fontweight='bold')
    
    # Plot 1: Vehicles per frame
    axes[0].plot(frame_nums, frame_counts, color='#2196F3', linewidth=1.5, alpha=0.8)
    axes[0].fill_between(frame_nums, frame_counts, alpha=0.2, color='#2196F3')
    axes[0].set_title('Vehicles Detected Per Frame')
    axes[0].set_xlabel('Frame Number')
    axes[0].set_ylabel('Vehicle Count')
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(y=np.mean(frame_counts), color='red',
                    linestyle='--', label=f'Avg: {np.mean(frame_counts):.1f}')
    axes[0].legend()
    
    # Plot 2: Class distribution bar chart
    if class_totals:
        classes = list(class_totals.keys())
        counts  = list(class_totals.values())
        bar_colors = [CLASS_COLORS.get(c, (128, 128, 128)) for c in classes]
        bar_colors = [(r/255, g/255, b/255) for b, g, r in bar_colors]
        bars = axes[1].bar(classes, counts, color=bar_colors, edgecolor='white', linewidth=1.5)
        axes[1].set_title('Vehicle Type Distribution')
        axes[1].set_xlabel('Vehicle Type')
        axes[1].set_ylabel('Total Count')
        for bar, count in zip(bars, counts):
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                         str(count), ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('detection_analytics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Analytics graph saved: detection_analytics.png')
else:
    print('⚠️  Pehle video process karo!')

## Step 8: Save Detection Data for Next Notebook

> Notebook 2 (Tracking) mein yeh data use hoga.

In [ ]:
import pickle

# Detection results save karo
with open('detection_results.pkl', 'wb') as f:
    pickle.dump({'all_detections': all_detections, 'stats': stats}, f)

print('✅ Detection results saved to: detection_results.pkl')
print('\n📌 Summary:')
print(f'   Total frames processed : {stats.get("total_frames_processed", 0)}')
print(f'   Total vehicle detections: {stats.get("total_detections", 0)}')
print(f'   Average per frame       : {stats.get("avg_vehicles_per_frame", 0):.2f}')
print('\n➡️  Ab Notebook 2 (Vehicle Tracking & Counting) chalao!')

---
## ✅ Notebook 1 Complete!

| Step | Status |
|------|--------|
| YOLOv8 model loaded | ✅ |
| Vehicle detection (car/bike/bus/truck) | ✅ |
| Bounding boxes + labels | ✅ |
| Output video saved | ✅ |
| Analytics graphs | ✅ |
| Results saved for Notebook 2 | ✅ |

**➡️ Next: `02_Vehicle_Tracking_Counting.ipynb`**